# LahjaLens — Audit MAC et expérimentation multi-datasets

Ce notebook audite le Moroccan Arabic Corpus, prépare trois configurations et exécute E0, E1 et E2 sur le cluster Spark.

E0 utilise le train AfriSenti officiel, E1 utilise MAC après nettoyage conservateur, et E2 combine les deux. Les lignes MAC mixed, vides et contradictoires sont isolées. Aucun test officiel n'est utilisé pour régler les paramètres.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict
import csv, hashlib, json, math, re, statistics, time, unicodedata, urllib.error, urllib.parse, urllib.request

from pyspark.sql import SparkSession, functions as F, types as T, Window
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.feature import CountVectorizer, IDF, NGram, Normalizer, RegexTokenizer, SQLTransformer, VectorAssembler

ROOT=Path("/opt/lahjalens"); DATA=ROOT/"data"
MAC_DIR=DATA/"raw/mac/source"; MAC_PATH=MAC_DIR/"MAC corpus.csv"; AF_DIR=DATA/"processed/afrisenti/v1"
BASE=DATA/"processed/mac_multidataset"; SEED=20261115; BOOTSTRAPS=2000
LABELS=["negative","neutral","positive"]; LABEL_MAP={x:i for i,x in enumerate(LABELS)}
RUN_ID=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")+"_"+hashlib.sha256(str(time.time_ns()).encode()).hexdigest()[:8]
OUT=BASE/RUN_ID
for child in ("audit","datasets","predictions","tables","training","evaluation","mlflow"): (OUT/child).mkdir(parents=True,exist_ok=False)
HDFS_BASE="hdfs://namenode:9000/data/lahjalens/models/sentiment/experiments/mac_multidataset/"+RUN_ID
MLFLOW_URL="http://mlflow:5000"; started=time.monotonic()
spark=(SparkSession.builder.appName("LahjaLens-MAC-"+RUN_ID).master("spark://spark-master:7077")
 .config("spark.hadoop.dfs.replication","2").config("spark.sql.shuffle.partitions","8")
 .config("spark.default.parallelism","4").config("spark.sql.csv.parser.columnPruning.enabled","false").getOrCreate())
spark.sparkContext.setLogLevel("WARN"); assert spark.sparkContext.master=="spark://spark-master:7077"
spark.sparkContext._jsc.hadoopConfiguration().set("dfs.replication","2")
CLUSTER={"master":spark.sparkContext.master,"application_id":spark.sparkContext.applicationId,"spark_version":spark.version,"python":__import__("sys").version.split()[0],"replication":spark.sparkContext._jsc.hadoopConfiguration().get("dfs.replication")}
print("RUN_ID",RUN_ID); print("OUT",OUT); print("CLUSTER",CLUSTER)

In [ ]:
URL_RE=re.compile(r"https?://\S+|www\.\S+",re.I); MENTION_RE=re.compile(r"@[\w_]+"); HASHTAG_RE=re.compile(r"#[\w_\u0600-\u06ff]+")
NEG={"ma","machi","mabghitch","mabghach","makaynch","makayna","la","lissa","lam","lan","not","never","no","pas","sans","ماشي","ما","مش","ليس","لا","لم","لن","مابقاش","ماكاينش"}

def rows(path):
    with path.open(encoding="utf-8-sig",newline="") as h:
        reader=csv.DictReader(h); return list(reader),reader.fieldnames
def clean_copy(text):
    return URL_RE.sub(" ",MENTION_RE.sub(" ",text or ""))
def script_group(text):
    ar=la=other=0
    for ch in clean_copy(text):
        if not ch.isalpha(): continue
        name=unicodedata.name(ch,"")
        if "ARABIC" in name: ar+=1
        elif "LATIN" in name: la+=1
        else: other+=1
    return "mixed" if ar and la else "arabic" if ar else "latin" if la else "other"
def arabizi(text):
    return any(re.search(r"[A-Za-z]",t) and re.search(r"[2356789]",t) for t in re.findall(r"[A-Za-z0-9]+",clean_copy(text)))
def norm_key(text):
    text=URL_RE.sub(" <url> ",MENTION_RE.sub(" <user> ",text or ""))
    return re.sub(r"\s+"," ",unicodedata.normalize("NFKC",text).casefold()).strip()
def negation(text):
    return any(x.casefold() in NEG for x in re.findall(r"[A-Za-z\u0600-\u06ff]+",text or ""))
def emoji(text):
    return any(unicodedata.category(c) in {"So","Sk"} and ord(c)>127 for c in text or "")
def commit_of(path):
    head=path/".git/HEAD"
    if not head.exists(): return "git_metadata_unavailable"
    v=head.read_text(encoding="utf-8").strip()
    return (path/".git"/v[5:]).read_text(encoding="utf-8").strip() if v.startswith("ref: ") and (path/".git"/v[5:]).exists() else v
def jwrite(rel,value):
    p=OUT/rel; p.parent.mkdir(parents=True,exist_ok=True); p.write_text(json.dumps(value,ensure_ascii=False,indent=2,allow_nan=False)+"\n",encoding="utf-8")
def twrite(rel,value):
    p=OUT/rel; p.parent.mkdir(parents=True,exist_ok=True); p.write_text(value,encoding="utf-8")
def profiles(data,text_col,label_col):
    groups=defaultdict(list)
    for r in data: groups[script_group(r.get(text_col,""))].append(r)
    out={}
    for g,items in sorted(groups.items()):
        chars=[len(r.get(text_col,"") or "") for r in items]; words=[len((r.get(text_col,"") or "").split()) for r in items]
        out[g]={"n":len(items),"labels":dict(Counter((r.get(label_col,"") or "").casefold() for r in items)),
          "mean_chars":round(statistics.mean(chars),4),"median_chars":statistics.median(chars),
          "mean_words":round(statistics.mean(words),4),"median_words":statistics.median(words),
          "mentions":sum(bool(MENTION_RE.search(r.get(text_col,"") or "")) for r in items),
          "urls":sum(bool(URL_RE.search(r.get(text_col,"") or "")) for r in items),
          "hashtags":sum(bool(HASHTAG_RE.search(r.get(text_col,"") or "")) for r in items),
          "emoji_rows":sum(emoji(r.get(text_col,"")) for r in items),
          "digit_rows":sum(any(c.isdigit() for c in r.get(text_col,"") or "") for r in items),
          "negation_rows":sum(negation(r.get(text_col,"")) for r in items),
          "arabizi_proxy":sum(arabizi(r.get(text_col,"")) for r in items)}
    return out
def sql_regex(expr,pattern,repl):
    return "regexp_replace("+expr+", r'"+pattern+"', '"+repl+"')"
def clean_expr(col):
    expr="lower("+col+")"
    for pat,repl in [(r"https?://\S+|www\.\S+"," "),(r"@[\w]+"," "),("[\u0640\u064b-\u065f\u0670]",""),("[\u0622\u0623\u0625\u0671]","\u0627"),(r"([a-z\u0621-\u064a])\1{2,}","$1$1"),(r"\s+"," ")]:
        expr=sql_regex(expr,pat,repl)
    return "trim("+expr+")"
cleaner=SQLTransformer(statement="SELECT *, "+clean_expr("comment_text")+" AS model_text FROM __THIS__")
script_udf=F.udf(script_group,T.StringType()); arabizi_udf=F.udf(arabizi,T.BooleanType()); key_udf=F.udf(norm_key,T.StringType())
def metadata(frame):
    return frame.withColumn("writing_script",script_udf("comment_text")).withColumn("arabizi_proxy",arabizi_udf("comment_text"))
def jmetrics(matrix):
    n=sum(map(sum,matrix)); pc={}
    for i,label in enumerate(LABELS):
        support=sum(matrix[i]); predicted=sum(row[i] for row in matrix); tp=matrix[i][i]
        pc[label]={"support":support,"precision":tp/predicted if predicted else 0.0,"recall":tp/support if support else 0.0,"f1":2*tp/(support+predicted) if support+predicted else 0.0}
    return {"rows":n,"accuracy":sum(matrix[i][i] for i in range(3))/n if n else None,"macro_f1":sum(x["f1"] for x in pc.values())/3 if n else None,"weighted_f1":sum(x["f1"]*x["support"] for x in pc.values())/n if n else None,"per_class":pc,"confusion_matrix":matrix,"labels":LABELS}
def frame_metrics(frame,group="all"):
    m=[[0]*3 for _ in range(3)]
    for r in frame.groupBy("label","prediction").count().collect(): m[int(r["label"])][int(r["prediction"])]=int(r["count"])
    result=jmetrics(m); result["group"]=group
    result["prediction_distribution"]={LABELS[int(r["prediction"])]:int(r["count"]) for r in frame.groupBy("prediction").count().collect()}
    result["frequent_errors"]=[{"true":LABELS[int(r["label"])],"predicted":LABELS[int(r["prediction"])],"count":int(r["count"])} for r in frame.where(F.col("label")!=F.col("prediction")).groupBy("label","prediction").count().orderBy(F.desc("count")).limit(10).collect()]
    return result
def hdfs_exists(uri):
    path=spark.sparkContext._jvm.org.apache.hadoop.fs.Path(uri); fs=path.getFileSystem(spark.sparkContext._jsc.hadoopConfiguration()); return bool(fs.exists(path))
def parquet(frame,rel):
    p=OUT/rel; frame.write.mode("errorifexists").parquet(str(p)); return str(p)
def csvout(frame,rel):
    p=OUT/rel; frame.write.mode("errorifexists").option("header",True).csv(str(p)); return str(p)
print("HELPERS_READY")

In [ ]:
# Phase A — audit MAC et comparaison AfriSenti
mac,mac_fields=rows(MAC_PATH); assert mac_fields==["tweets","type","class"]
af={}
for split in ("train","dev","test"):
    af[split],fields=rows(AF_DIR/(split+".csv"))
    assert all(r.get("sentiment") in LABEL_MAP for r in af[split])
mac_hash=hashlib.sha256(MAC_PATH.read_bytes()).hexdigest(); mac_keys=defaultdict(list)
for r in mac: mac_keys[norm_key(r.get("tweets",""))].append(r)
dup_groups=[g for g in mac_keys.values() if len(g)>1]
conflicts=[g for g in dup_groups if len({(r.get("type") or "").casefold() for r in g})>1]
af_all=[dict(r,_split=s) for s in af for r in af[s]]; af_keys=defaultdict(list)
for r in af_all: af_keys[norm_key(r.get("comment_text",""))].append(r)
overlaps=[]
for k,items in mac_keys.items():
    for a in items:
        for b in af_keys.get(k,[]): overlaps.append({"mac_type":a.get("type"),"af_split":b["_split"],"af_id":b.get("record_id"),"af_sentiment":b.get("sentiment"),"key_sha256":hashlib.sha256(k.encode()).hexdigest()})
mac_labels=Counter((r.get("type") or "").casefold() for r in mac); mac_classes=Counter((r.get("class") or "").casefold() for r in mac)
audit={"run_id":RUN_ID,"source":{"repository":"https://github.com/LeMGarouani/MAC","commit":commit_of(MAC_DIR),"path":str(MAC_PATH),"sha256":mac_hash,"bytes":MAC_PATH.stat().st_size,"repository_license_file_present":False,"license_note":"README says open/free; no LICENSE file at audited commit; raw redistribution is not assumed authorized."},
"mac":{"rows":len(mac),"fields":mac_fields,"missing":{f:sum(not (r.get(f) or "").strip() for r in mac) for f in mac_fields},"labels":dict(mac_labels),"language_classes":dict(mac_classes),"empty_text_rows":sum(not (r.get("tweets") or "").strip() for r in mac),"mixed_sentiment_rows":mac_labels.get("mixed",0),"exact_duplicate_rows":len(mac)-len({r.get("tweets","") for r in mac}),"normalized_duplicate_rows":len(mac)-len(mac_keys),"duplicate_groups":len(dup_groups),"conflicting_groups":len(conflicts),"conflicting_rows":sum(len(x) for x in conflicts),"three_class_nonempty_rows":sum(bool((r.get("tweets") or "").strip()) and (r.get("type") or "").casefold() in LABEL_MAP for r in mac),"unique_single_label_rows":sum(1 for g in mac_keys.values() if g[0].get("tweets","").strip() and (g[0].get("type") or "").casefold() in LABEL_MAP and len({(r.get("type") or "").casefold() for r in g})==1),"by_script":profiles(mac,"tweets","type"),"script_counts":dict(Counter(script_group(r.get("tweets","")) for r in mac)),"arabizi_proxy_rows":sum(arabizi(r.get("tweets","")) for r in mac)},
"afrisenti":{s:{"rows":len(af[s]),"sha256":hashlib.sha256((AF_DIR/(s+".csv")).read_bytes()).hexdigest(),"labels":dict(Counter(r["sentiment"] for r in af[s])),"by_script":profiles(af[s],"comment_text","sentiment")} for s in af},
"cross_dataset_normalized_overlap_pairs":len(overlaps),"cross_dataset_overlap_by_split":dict(Counter(r["af_split"] for r in overlaps)),"cross_dataset_label_conflicts":sum(r["mac_type"]!=r["af_sentiment"] for r in overlaps)}
jwrite("audit/mac_audit.json",audit); jwrite("audit/cross_dataset_overlaps.json",overlaps)
m=audit["mac"]; report=["# Audit MAC — LahjaLens","",f"Run : {RUN_ID}",f"Source commit : {audit['source']['commit']}",f"SHA-256 : {mac_hash}","",
"## Licence","", "Le dépôt audité contient le corpus, un lexique et un README, mais aucun fichier LICENSE. Le README décrit le corpus comme open/free, ce qui ne suffit pas à établir une licence de redistribution brute. MAC reste donc local et ne sera pas publié dans LahjaLens sans clarification.","",
"## Résultats observés","",f"- {m['rows']} lignes brutes et {m['empty_text_rows']} textes vides.",f"- Labels : {m['labels']}. Le label mixed est exclu, jamais converti en neutral.",f"- Classes linguistiques source : {m['language_classes']}. La valeur diatectal est conservée comme anomalie source.",f"- {m['normalized_duplicate_rows']} lignes appartiennent à des doublons normalisés ; {m['conflicting_groups']} groupes ont des labels contradictoires.",f"- {m['unique_single_label_rows']} textes uniques à label unique sont disponibles avant partitionnement.",f"- Recouvrement normalisé avec AfriSenti : {len(overlaps)} paire.",f"- Proxy Arabizi heuristique dans MAC : {m['arabizi_proxy_rows']}; ce nombre n'est pas une annotation linguistique.","",
"## Décision de préparation","", "E1/E2 suppriment les lignes vides, mixed et les groupes contradictoires. Les doublons à label unique sont réduits à une ligne canonique. Les partitions sont affectées par hachage stable du texte normalisé. Les vocabulaires, IDF et poids sont appris uniquement sur chaque train."] 
twrite("audit/MAC_AUDIT.md","\n".join(report)+"\n")
print("MAC_AUDIT",json.dumps(audit["mac"],ensure_ascii=False)); print("OVERLAPS",len(overlaps))

In [ ]:
# Phase B — préparation et prévention du leakage
schema=T.StructType([T.StructField("tweets",T.StringType(),True),T.StructField("type",T.StringType(),True),T.StructField("class",T.StringType(),True)])
raw=(spark.read.schema(schema).options(header=True,enforceSchema=False,mode="FAILFAST",encoding="UTF-8",quote='"',escape='"',multiLine=True).csv(str(MAC_PATH)))
mac0=raw.select(F.col("tweets").alias("comment_text"),F.lower(F.trim("type")).alias("sentiment_name"),F.trim("class").alias("language_class")).withColumn("_key",key_udf("comment_text"))
mac_mixed=mac0.where(F.col("sentiment_name")=="mixed")
mac_valid=mac0.where(F.col("sentiment_name").isin(LABELS)&(F.length(F.trim("comment_text"))>0)).withColumn("label",F.when(F.col("sentiment_name")=="negative",0).when(F.col("sentiment_name")=="neutral",1).otherwise(2))
groups=mac_valid.groupBy("_key").agg(F.collect_set("sentiment_name").alias("_labels"))
conflict_keys=groups.where(F.size("_labels")>1).select("_key"); mac_conflicts=mac_valid.join(conflict_keys,"_key")
single=mac_valid.join(groups.where(F.size("_labels")==1).select("_key"),"_key")
w=Window.partitionBy("_key").orderBy(F.sha2(F.concat_ws("\u0001","comment_text","sentiment_name","language_class"),256))
mac_unique=(single.withColumn("_rn",F.row_number().over(w)).where(F.col("_rn")==1).drop("_rn")
 .withColumn("record_id",F.concat(F.lit("mac:v1:"),F.sha2("_key",256))).withColumn("source_dataset",F.lit("mac"))
 .withColumn("source_split",F.lit("pending")).withColumn("cleaning_version",F.lit("shared_clean_v1")))
mac_unique=metadata(mac_unique)
def af_frame(split):
    sch=T.StructType([T.StructField("record_id",T.StringType(),False),T.StructField("comment_text",T.StringType(),False),T.StructField("sentiment",T.StringType(),False),T.StructField("label",T.StringType(),False),T.StructField("source_dataset",T.StringType(),False),T.StructField("source_split",T.StringType(),False),T.StructField("language",T.StringType(),True),T.StructField("cleaning_version",T.StringType(),True)])
    x=spark.read.schema(sch).options(header=True,enforceSchema=False,mode="FAILFAST",encoding="UTF-8",quote='"',escape='"',multiLine=True).csv(str(AF_DIR/(split+".csv")))
    return metadata(x.withColumn("label",F.col("label").cast("int")).withColumn("sentiment_name",F.col("sentiment")).withColumn("_key",key_udf("comment_text")))
AF={s:af_frame(s) for s in ("train","dev","test")}
reserved=AF["dev"].select("_key").unionByName(AF["test"].select("_key")).distinct()
mac_reserved=mac_unique.join(reserved,"_key"); assert mac_reserved.count()==0
bucket=F.pmod(F.xxhash64(F.col("_key"),F.lit(SEED)),F.lit(100))
mac_split=mac_unique.withColumn("_bucket",bucket).withColumn("source_split",F.when(F.col("_bucket")<80,"train").when(F.col("_bucket")<90,"dev").otherwise("test")).drop("_bucket")
MS={s:mac_split.where(F.col("source_split")==s).cache() for s in ("train","dev","test")}
E0_TRAIN,E0_DEV,E0_TEST=AF["train"].cache(),AF["dev"].cache(),AF["test"].cache()
E1_TRAIN,E1_DEV,E1_TEST=MS["train"],MS["dev"],MS["test"]
E2_TRAIN=E0_TRAIN.unionByName(E1_TRAIN,allowMissingColumns=True).cache()
summary={"mac_raw":raw.count(),"mac_mixed":mac_mixed.count(),"mac_conflicts":mac_conflicts.count(),"mac_reserved_overlap":mac_reserved.count(),"mac_unique":mac_unique.count(),"mac_splits":{s:MS[s].count() for s in MS},"e0_train":E0_TRAIN.count(),"e1_train":E1_TRAIN.count(),"e2_train":E2_TRAIN.count(),"labels":{x:{"E0":E0_TRAIN.where(F.col("sentiment_name")==x).count(),"E1":E1_TRAIN.where(F.col("sentiment_name")==x).count(),"E2":E2_TRAIN.where(F.col("sentiment_name")==x).count()} for x in LABELS},"scripts":{x:{r["writing_script"]:r["count"] for r in frame.groupBy("writing_script").count().collect()} for x,frame in {"E0":E0_TRAIN,"E1":E1_TRAIN,"E2":E2_TRAIN}.items()}}
for name,frame in {"mac_excluded_mixed":mac_mixed,"mac_excluded_conflicts":mac_conflicts,"mac_excluded_reserved":mac_reserved,"afrisenti_train":E0_TRAIN,"afrisenti_dev":E0_DEV,"afrisenti_test":E0_TEST,"mac_train":E1_TRAIN,"mac_dev":E1_DEV,"mac_test":E1_TEST,"combined_train":E2_TRAIN}.items(): parquet(frame,"datasets/"+name)
jwrite("audit/partition_summary.json",summary); print("PARTITIONS",json.dumps(summary,ensure_ascii=False))

In [ ]:
# Phase C — training E0, E1, E2
def feature_pipeline():
    stages=[cleaner.copy(),SQLTransformer(statement="""SELECT *, flatten(transform(array(3,4,5), n -> CASE WHEN length(model_text)>=n THEN transform(sequence(1,length(model_text)-n+1),i -> substring(model_text,i,n)) ELSE cast(array() AS array<string>) END)) AS char_tokens FROM __THIS__"""),CountVectorizer(inputCol="char_tokens",outputCol="char_tf",vocabSize=80000,minDF=2.0),IDF(inputCol="char_tf",outputCol="char_idf",minDocFreq=2),Normalizer(inputCol="char_idf",outputCol="char_norm",p=2.0),RegexTokenizer(inputCol="model_text",outputCol="words",pattern=r"\s+",minTokenLength=1),NGram(n=2,inputCol="words",outputCol="bigrams"),SQLTransformer(statement="SELECT *, concat(words,bigrams) AS word_tokens FROM __THIS__"),CountVectorizer(inputCol="word_tokens",outputCol="word_tf",vocabSize=50000,minDF=2.0),IDF(inputCol="word_tf",outputCol="word_idf",minDocFreq=2),Normalizer(inputCol="word_idf",outputCol="word_norm",p=2.0),VectorAssembler(inputCols=["word_norm","char_norm"],outputCol="combined"),Normalizer(inputCol="combined",outputCol="features",p=2.0),LogisticRegression(featuresCol="features",labelCol="label",weightCol="sample_weight",family="multinomial",regParam=0.1,standardization=True,maxIter=300,tol=1e-6)]
    return Pipeline(stages=stages)
def weights(frame):
    counts=frame.groupBy("label").count(); assert {int(x["label"]) for x in counts.collect()}=={0,1,2}; n=frame.count()
    w=counts.withColumn("sample_weight",F.lit(float(n)/3.0)/F.col("count")).drop("count")
    return frame.join(w,"label"),[x.asDict() for x in w.orderBy("label").collect()]
TRAIN={"E0":E0_TRAIN,"E1":E1_TRAIN,"E2":E2_TRAIN}; MODELS={}; INFO={}
for name,frame in TRAIN.items():
    t=time.monotonic(); weighted,ww=weights(frame); fitted=feature_pipeline().fit(weighted); path=HDFS_BASE+"/"+name.lower()
    if hdfs_exists(path): raise RuntimeError("Refusing HDFS overwrite: "+path)
    fitted.write().save(path); MODELS[name]=fitted; stage=fitted.stages[-1]
    conv={"iterations":int(stage.summary.totalIterations) if hasattr(stage,"summary") else None,"objective_history":[float(x) for x in stage.summary.objectiveHistory] if hasattr(stage,"summary") else []}
    INFO[name]={"dataset":{"E0":"AfriSenti","E1":"MAC","E2":"AfriSenti+MAC"}[name],"train_rows":frame.count(),"weights":ww,"regParam":0.1,"standardization":True,"maxIter":300,"tol":1e-6,"hdfs_path":path,"training_seconds":round(time.monotonic()-t,2),"convergence":conv,"spark_master":spark.sparkContext.master,"application_id":spark.sparkContext.applicationId}
    print("TRAINED",name,INFO[name])
jwrite("training/model_info.json",INFO)

In [ ]:
# Phase D — metrics and paired bootstrap
import numpy as np
EVAL={"afrisenti_test":E0_TEST,"mac_test":E1_TEST}; PRED={}; RESULTS={}
def group_intervals(frame,seed):
    rows=frame.select("label","prediction").collect()
    if not rows: return {"macro_f1_ci95":[None,None],"negative_recall_ci95":[None,None],"bootstrap_repetitions":BOOTSTRAPS,"seed":seed}
    y=np.asarray([int(r["label"]) for r in rows],dtype=np.int64); p=np.asarray([int(r["prediction"]) for r in rows],dtype=np.int64); rng=np.random.default_rng(seed); vals=[]
    for _ in range(BOOTSTRAPS):
        idx=rng.integers(0,len(y),len(y)); matrix=np.bincount(3*y[idx]+p[idx],minlength=9).reshape(3,3); z=jmetrics(matrix); vals.append((z["macro_f1"],z["per_class"]["negative"]["recall"]))
    vals.sort(); lo=int(.025*len(vals)); hi=int(.975*len(vals))-1; negative=sorted(v[1] for v in vals)
    return {"macro_f1_ci95":[float(vals[lo][0]),float(vals[hi][0])],"negative_recall_ci95":[float(negative[lo]),float(negative[hi])],"bootstrap_repetitions":BOOTSTRAPS,"seed":seed}
def group_metrics(frame,group,seed):
    result=frame_metrics(frame,group); result["confidence_intervals_95"]=group_intervals(frame,seed); return result
for en,frame in EVAL.items():
    ids={x["record_id"] for x in frame.select("record_id").collect()}
    for name,model in MODELS.items():
        p=(model.transform(frame).select("record_id","label","writing_script","arabizi_proxy",F.col("prediction").cast("int").alias("prediction")).cache())
        assert {x["record_id"] for x in p.select("record_id").collect()}==ids
        PRED[(en,name)]=p; csvout(p,"predictions/"+en+"_"+name)
        RESULTS[(en,name)]={"all":group_metrics(p,"all",SEED+len(RESULTS))}
        for g in ("arabic","latin","mixed","other"):
            RESULTS[(en,name)][g]=group_metrics(p.where(F.col("writing_script")==g),g,SEED+len(RESULTS)+len(g))
        RESULTS[(en,name)]["arabizi_proxy"]=group_metrics(p.where(F.col("arabizi_proxy")),"arabizi_proxy",SEED+len(RESULTS)+99)
def arrays(en):
    yrows=EVAL[en].select("record_id","label").orderBy("record_id").collect(); y=[int(x["label"]) for x in yrows]; a={}
    for name in MODELS:
        prows=PRED[(en,name)].select("record_id","prediction").orderBy("record_id").collect(); assert [x["record_id"] for x in yrows]==[x["record_id"] for x in prows]; a[name]=[int(x["prediction"]) for x in prows]
    return y,a
def boot(y,left,right,seed):
    import numpy as np
    y=np.asarray(y,dtype=np.int64); left=np.asarray(left,dtype=np.int64); right=np.asarray(right,dtype=np.int64); rng=np.random.default_rng(seed); values=[]
    def score(p):
        return jmetrics(np.bincount(3*y[idx]+p[idx],minlength=9).reshape(3,3))["macro_f1"]
    for _ in range(BOOTSTRAPS):
        idx=rng.integers(0,len(y),len(y)); values.append(score(right)-score(left))
    values.sort(); return {"observed":float(jmetrics(np.bincount(3*y+right,minlength=9).reshape(3,3))["macro_f1"]-jmetrics(np.bincount(3*y+left,minlength=9).reshape(3,3))["macro_f1"]),"ci_low":float(values[int(.025*BOOTSTRAPS)]),"ci_high":float(values[int(.975*BOOTSTRAPS)-1]),"n":len(y),"repetitions":BOOTSTRAPS,"seed":seed}
BOOT=[]
for en in EVAL:
    y,a=arrays(en)
    for left,right in (("E0","E1"),("E0","E2"),("E1","E2")):
        row=boot(y,a[left],a[right],SEED); row.update({"evaluation_set":en,"comparison":right+"-"+left}); BOOT.append(row)
jwrite("evaluation/results.json",{"metrics":{en:{name:RESULTS[(en,name)] for name in MODELS} for en in EVAL},"paired_bootstrap":BOOT})
print("BOOTSTRAP",json.dumps(BOOT,ensure_ascii=False))

In [ ]:
# Phase E — MLflow, report and manifest
def mlreq(endpoint,payload):
    req=urllib.request.Request(MLFLOW_URL+endpoint,data=json.dumps(payload).encode(),headers={"Content-Type":"application/json"},method="POST")
    with urllib.request.urlopen(req,timeout=10) as response: return json.loads(response.read().decode())
def mlget(endpoint,params):
    url=MLFLOW_URL+endpoint+"?"+urllib.parse.urlencode(params)
    with urllib.request.urlopen(url,timeout=10) as response: return json.loads(response.read().decode())
MLFLOW={"url":MLFLOW_URL,"runs":[],"error":None}
try:
    exp=mlget("/api/2.0/mlflow/experiments/get-by-name",{"experiment_name":"LahjaLens-MAC-multidataset"})["experiment"]
except urllib.error.HTTPError as error:
    if error.code!=404: raise
    exp=mlreq("/api/2.0/mlflow/experiments/create",{"name":"LahjaLens-MAC-multidataset"})
for name,info in INFO.items():
    try:
        run=mlreq("/api/2.0/mlflow/runs/create",{"experiment_id":exp["experiment_id"],"start_time":int(time.time()*1000),"tags":[{"key":"strategy","value":name},{"key":"run_id","value":RUN_ID}]})
        rid=run["run"]["info"]["run_id"]
        for k,v in {"train_rows":info["train_rows"],"regParam":info["regParam"],"maxIter":info["maxIter"],"seed":SEED}.items(): mlreq("/api/2.0/mlflow/runs/log-parameter",{"run_id":rid,"key":k,"value":str(v)})
        for en in EVAL:
            m=RESULTS[(en,name)]["all"]
            for k,v in {"macro_f1":m["macro_f1"],"accuracy":m["accuracy"],"weighted_f1":m["weighted_f1"],"negative_recall":m["per_class"]["negative"]["recall"]}.items(): mlreq("/api/2.0/mlflow/runs/log-metric",{"run_id":rid,"key":en+"_"+k,"value":float(v),"timestamp":int(time.time()*1000),"step":0})
        mlreq("/api/2.0/mlflow/runs/update",{"run_id":rid,"status":"FINISHED","end_time":int(time.time()*1000)}); MLFLOW["runs"].append({"strategy":name,"run_id":rid})
    except Exception as error: MLFLOW["error"]=str(error); print("MLFLOW_WARNING",name,repr(error))
def pct(v): return "n/a" if v is None else f"{100*v:.2f}%"
def svg_escape(value): return str(value).replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")
def macro_svg():
    width,height,left,top,bottom=980,520,90,55,80; plot_h=height-top-bottom; colors={"E0":"#2563eb","E1":"#f59e0b","E2":"#16a34a"}; parts=[f"<svg xmlns='http://www.w3.org/2000/svg' width='{width}' height='{height}' viewBox='0 0 {width} {height}'>","<rect width='100%' height='100%' fill='white'/>","<text x='490' y='28' text-anchor='middle' font-family='Arial' font-size='20' font-weight='bold'>Macro-F1 par strategie et jeu d'evaluation</text>"]
    for tick in range(0,11,2):
        y=top+plot_h-(tick/10)*plot_h; parts += [f"<line x1='{left}' y1='{y:.1f}' x2='{width-35}' y2='{y:.1f}' stroke='#d1d5db'/>",f"<text x='{left-12}' y='{y+5:.1f}' text-anchor='end' font-family='Arial' font-size='12'>{tick*10}%</text>"]
    group_w=(width-left-35)/len(EVAL); bar_w=group_w/(len(MODELS)+1)
    for gi,en in enumerate(EVAL):
        gx=left+gi*group_w; parts.append(f"<text x='{gx+group_w/2:.1f}' y='{height-42}' text-anchor='middle' font-family='Arial' font-size='14'>{svg_escape(en)}</text>")
        for si,name in enumerate(MODELS):
            value=RESULTS[(en,name)]["all"]["macro_f1"]; x=gx+(si+0.5)*bar_w; y=top+plot_h-value*plot_h; parts += [f"<rect x='{x:.1f}' y='{y:.1f}' width='{bar_w*.78:.1f}' height='{(value*plot_h):.1f}' fill='{colors[name]}'/>",f"<text x='{x+bar_w*.39:.1f}' y='{y-7:.1f}' text-anchor='middle' font-family='Arial' font-size='11'>{value*100:.1f}%</text>"]
    for si,name in enumerate(MODELS):
        x=width-250+si*78; parts += [f"<rect x='{x}' y='{height-28}' width='14' height='14' fill='{colors[name]}'/>",f"<text x='{x+19}' y='{height-16}' font-family='Arial' font-size='12'>{name}</text>"]
    parts.append("</svg>"); return "\n".join(parts)
def confusion_svg(en):
    width,height=1140,410; parts=[f"<svg xmlns='http://www.w3.org/2000/svg' width='{width}' height='{height}' viewBox='0 0 {width} {height}'>","<rect width='100%' height='100%' fill='white'/>",f"<text x='570' y='28' text-anchor='middle' font-family='Arial' font-size='20' font-weight='bold'>Matrices de confusion — {svg_escape(en)}</text>"]; labels=['neg','neu','pos']
    for mi,name in enumerate(MODELS):
        matrix=RESULTS[(en,name)]["all"]["confusion_matrix"]; x0=55+mi*365; y0=95; cell=55; vmax=max(max(row) for row in matrix) or 1; parts.append(f"<text x='{x0+110}' y='65' text-anchor='middle' font-family='Arial' font-size='16' font-weight='bold'>{name}</text>")
        for i in range(3):
            parts.append(f"<text x='{x0-12}' y='{y0+i*cell+34}' text-anchor='end' font-family='Arial' font-size='12'>{labels[i]}</text>")
            parts.append(f"<text x='{x0+i*cell+27}' y='{y0-10}' text-anchor='middle' font-family='Arial' font-size='12'>{labels[i]}</text>")
            for j in range(3):
                count=matrix[i][j]; shade=255-int(190*count/vmax); fill=f"rgb({shade},{shade},{255})"; parts += [f"<rect x='{x0+j*cell}' y='{y0+i*cell}' width='{cell}' height='{cell}' fill='{fill}' stroke='white'/>",f"<text x='{x0+j*cell+27}' y='{y0+i*cell+34}' text-anchor='middle' font-family='Arial' font-size='14'>{count}</text>"]
    parts += ["<text x='570' y='385' text-anchor='middle' font-family='Arial' font-size='12'>Colonnes = prédiction · lignes = vérité</text>","</svg>"]; return "\n".join(parts)
twrite("plots/macro_f1_comparison.svg",macro_svg())
for en in EVAL: twrite("plots/confusion_"+en+".svg",confusion_svg(en))
report=["# LahjaLens — MAC et expérimentation multi-datasets","",f"Run {RUN_ID}","",f"Cluster {CLUSTER}","", "Les résultats sont exploratoires. Les données MAC brutes restent locales.", "","## Audit","",f"Voir audit/MAC_AUDIT.md. MAC brut {summary['mac_raw']}, mixed exclus {summary['mac_mixed']}, conflits exclus {summary['mac_conflicts']}, uniques utilisées {summary['mac_unique']}.","", "## Résultats","", "| Expérience | Dataset | Évaluation | Macro-F1 | F1 négatif | Rappel négatif Latin |","|---|---|---|---:|---:|---:|"]
for en in EVAL:
    for name in MODELS:
        m=RESULTS[(en,name)]["all"]; lat=RESULTS[(en,name)]["latin"]
        report.append(f"| {name} | {'AfriSenti' if name=='E0' else 'MAC' if name=='E1' else 'AfriSenti + MAC'} | {en} | {pct(m['macro_f1'])} | {pct(m['per_class']['negative']['f1'])} | {pct(lat['per_class']['negative']['recall'])} |")
report += ["","## Bootstrap apparié",""]
for row in BOOT: report.append(f"- {row['evaluation_set']} {row['comparison']} : gain {100*row['observed']:+.2f} points, IC95 [{100*row['ci_low']:+.2f}; {100*row['ci_high']:+.2f}] points.")
report += ["","## Intervalles par écriture","","Les IC95% ci-dessous sont des bootstrap non appariés à graine fixe ; les groupes vides sont signalés comme non estimables."]
for en in EVAL:
    for name in MODELS:
        for group in ("arabic","latin","mixed","other","arabizi_proxy"):
            item=RESULTS[(en,name)][group]; ci=item["confidence_intervals_95"]; report.append(f"- {en} / {name} / {group}: support {item['rows']}, Macro-F1 {pct(item['macro_f1'])}, IC95 [{pct(ci['macro_f1_ci95'][0])}; {pct(ci['macro_f1_ci95'][1])}], rappel négatif {pct(item['per_class']['negative']['recall'])}, IC95 [{pct(ci['negative_recall_ci95'][0])}; {pct(ci['negative_recall_ci95'][1])}].")
report += ["","## Visualisations","","- [Comparaison Macro-F1](plots/macro_f1_comparison.svg)","- [Matrices AfriSenti](plots/confusion_afrisenti_test.svg)","- [Matrices MAC](plots/confusion_mac_test.svg)","- Les distributions de predictions et les erreurs frequentes sont dans evaluation/results.json.","","## Limites","","- MAC est majoritairement en ecriture arabe et n'est pas un correctif direct du deficit Latin/Arabizi.","- mixed est un sentiment exclu, pas un synonyme de neutral.","- Aucun score Hespress independant n'est disponible sans annotation humaine.","","## MLflow","",str(MLFLOW)]
twrite("comparison_report.md","\n".join(report)+"\n"); jwrite("mlflow/status.json",MLFLOW)
manifest={"status":"completed","run_id":RUN_ID,"created_at":datetime.now(timezone.utc).isoformat(),"cluster":CLUSTER,"source":{"repository":"https://github.com/LeMGarouani/MAC","commit":commit_of(MAC_DIR),"sha256":hashlib.sha256(MAC_PATH.read_bytes()).hexdigest()},"partitions":summary,"models":INFO,"evaluation_sets":{k:v.count() for k,v in EVAL.items()},"elapsed_seconds":round(time.monotonic()-started,2)}
jwrite("run_manifest.json",manifest); print("COMPLETED",RUN_ID); print("REPORT",OUT/"comparison_report.md")